# Environment setup

This notebook sets up everything you need for the hackathon:

1. Create a **virtual environment** for the project
2. Install the required libraries from `requirements.txt`
3. Register the environment as a **Jupyter kernel**
4. Check that the **GPU** is visible from PyTorch
5. **Download** the dataset

Run the cells from top to bottom with `Shift + Enter`.

<!-- TODO(organisers): add any cluster-specific notes here (modules to load, shared data paths, quotas, ...). -->

> **On BinderHub?** Most of the environment is already set up for you. Stay on the default kernel, run the cell
> below to add the few missing libraries, then skip straight to **Part 2**.
>
> **Not on BinderHub** (e.g. your own laptop or a cluster)? Skip the cell below. Instead, expand the collapsed **Part 1**
> section (click the arrow next to its heading, or the `...` placeholders to unhide the code cells) and follow its
> instructions to create the environment before continuing.

In [ ]:
# BinderHub only: install the libraries that aren't in the image yet (quantization and FPGA conversion).
# This lasts for the current session; run it again whenever you start a new BinderHub session.
%pip install "brevitas>=0.13" "hls4ml>=1.3" onnx onnxoptimizer qonnx

## Part 1: Create the environment

> Run Part 1 with the **default kernel** that JupyterLab gives you. You'll switch to the new kernel in Part 2.

<!-- TODO(organisers): short explanation of why we isolate dependencies per project. -->

First, let's see which Python we're running and where the repository is.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()  # this notebook lives in the repository root
VENV = REPO / ".venv"
PY = VENV / "bin" / "python"

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("Repository root:  ", REPO)
assert (REPO / "requirements.txt").exists(), "Open this notebook from the root folder of the repository."

### 1.1 Create the environment

We create it in `.venv/` inside the repository, with the same Python as the default kernel (3.10 or newer).

The `!` at the start of a line tells Jupyter to run it as a **shell command**. `{VENV}` inserts the value of the Python variable.

In [ ]:
assert sys.version_info >= (3, 10), "Python 3.10 or newer is needed. Ask an organiser!"
!rm -rf {VENV}
!{sys.executable} -m venv {VENV}
!{PY} --version
assert PY.exists(), f"Creating the environment failed: {PY} is missing. Check the output above."

### 1.2 Install the dependencies

This takes a few minutes. PyTorch is the largest download.

In [ ]:
!{PY} -m pip install --upgrade pip
!{PY} -m pip install -r {REPO}/requirements.txt

### 1.3 Register the environment as a Jupyter kernel

Jupyter can only use your new environment once it's registered as a *kernel*.

In [ ]:
!{PY} -m ipykernel install --user --name adatl1 --display-name "Python (adatl1)"
!jupyter kernelspec list

### 1.4 Switch kernel

Now switch this notebook over to the new environment:

**Kernel → Change Kernel… → `Python (adatl1)`**

(You may need to refresh the browser tab before it appears.) Then carry on with Part 2 below. **Don't re-run Part 1.**

<details>
<summary>Prefer the terminal? Click here for the equivalent commands.</summary>

```bash
cd hackathon_ADatL1
python -m venv .venv
.venv/bin/python -m pip install --upgrade pip
.venv/bin/python -m pip install -r requirements.txt
.venv/bin/python -m ipykernel install --user --name adatl1 --display-name "Python (adatl1)"
```
</details>

## Part 2: Verify the environment

> From here on, the kernel name in the top-right corner should say **Python (adatl1)**.

In [ ]:
import sys
from pathlib import Path

print(sys.executable)
print(sys.version)
if Path(".venv").exists():
    assert ".venv" in sys.executable, "You are not on the adatl1 kernel yet: Kernel -> Change Kernel -> Python (adatl1)"

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*fast_hadamard_transform.*")  # optional Brevitas speed-up, not needed

import numpy, pandas, pyarrow, awkward, sklearn, matplotlib, yaml
import torch, brevitas, hls4ml, onnx, qonnx

for mod in (numpy, pandas, pyarrow, awkward, sklearn, matplotlib, yaml, torch, brevitas, hls4ml, onnx, qonnx):
    print(f"{mod.__name__:12s} {getattr(mod, '__version__', 'unknown')}")

### 2.1 GPU check

<!-- TODO(organisers): describe the GPU resources available (type, how many per team, sharing etiquette). -->

In [ ]:
import os, glob, subprocess

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return (r.stdout or r.stderr).strip() or "(nothing)"

print("GPU device files:      ", glob.glob("/dev/nvidia*") or "none")
print("NVIDIA driver version: ", run("cat /proc/driver/nvidia/version 2>&1 | head -1"))
print("libcuda on system:     ", run("ldconfig -p 2>/dev/null | grep -i libcuda | head -3"))
print("nvidia-smi locations:  ", glob.glob("/usr/local/nvidia/bin/*") + glob.glob("/usr/bin/nvidia*") or "none")
for var in ("NVIDIA_VISIBLE_DEVICES", "CUDA_VISIBLE_DEVICES", "NVIDIA_DRIVER_CAPABILITIES"):
    print(f"{var:23s}", os.environ.get(var, "unset"))

print("\n--- hub / session info ---")
for k, v in sorted(os.environ.items()):
    if k.startswith(("JUPYTERHUB", "BINDER", "REPO_", "KUBERNETES_SERVICE_HOST", "MEM_", "CPU_")) and "TOKEN" not in k and "API" not in k:
        print(f"{k:23s} {v}")
print("CPUs:", os.cpu_count())
print("Memory:", run("free -h | sed -n 2p"))
print("Hostname:", run("hostname"))

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  [{i}] {torch.cuda.get_device_name(i)}")
    device = torch.device("cuda")
else:
    print("No GPU found; falling back to CPU. Ask an organiser if you expected one!")
    device = torch.device("cpu")

# Quick smoke test: a matrix multiplication on the device
a = torch.randn(4096, 4096, device=device)
print("matmul OK:", (a @ a).shape, "on", device)

## Part 3: Download the data

The dataset is hosted on the [Hugging Face Hub](https://huggingface.co/datasets/CERN/anomaly_detection_cmsl1t).
The full record is tens of GB, so we only download what the config files in `config/` need.

| Config | Downloads | Approx. size |
|---|---|---|
| `config/baseline.yml` | one zero-bias run + 4 signals | ~1.1 GB |
| `config/combined.yml` | both zero-bias runs + 4 signals | ~2 GB |
| `config/simulation.yml` | simulated zero bias + 4 signals | ~0.4 GB |

Files go to `data/` in the repository. If the organisers have put a shared copy somewhere, change
`root:` in the config files to that path instead and skip the download.

<!-- TODO(organisers): mention any shared data path. -->

In [ ]:
from dataloader import download_config

download_config("config/baseline.yml")

In [ ]:
# Optional: the other experiments
# download_config("config/combined.yml")
# download_config("config/simulation.yml")

In [ ]:
# Sanity check: load a few events
import dataset

events = dataset.load_events("ZB_run396102", "valid", max_events=1000)
print(len(events), "events loaded; fields:", events.fields)

**You're all set!** Continue with [`explore_data.ipynb`](explore_data.ipynb).